# Statistical analysis of `gender_given` prompt cases

I want to be determine if there a behavioural difference across models when gender is specified to be one binary value or another. For this I am planning a logistic regression anaylsis to see if diversity and entropy metrics are informative of gender/have significant predictive power over the label.

## Load data

In [17]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent
results_dir = root_dir / "data" / "olmo7b_results"
given_file = results_dir / "given_occupation_gender_metrics_olmo7b_temp_results_given.json"

import pandas as pd
df = pd.read_json(given_file)
df.head()

,occupation,gender,model_key,self_bleu_mean,self_bleu_std,self_bleu_median,self_bleu_count,semantic_div_mean,semantic_div_std,semantic_div_median,semantic_div_count,avg_mean_entropy_mean,avg_mean_entropy_std,avg_mean_entropy_median,avg_mean_entropy_count,response_count
0,artist,female,base,0.370723,0.109226,0.364182,8,0.310821,0.060632,0.309728,8,0.761605,0.559095,0.631648,8,80
1,artist,female,dpo,0.243857,0.105998,0.256526,8,0.351134,0.072555,0.345403,8,0.909978,0.670376,0.782070,8,80
2,artist,female,rlvr,0.281721,0.127461,0.272629,8,0.390963,0.097259,0.401014,8,0.830477,0.625345,0.693989,8,80
3,artist,female,sft,0.300213,0.136358,0.292288,8,0.366194,0.122224,0.391833,8,0.892600,0.708250,0.655094,8,80
4,artist,male,base,0.373313,0.141582,0.382950,8,0.257849,0.132448,0.247073,8,0.825648,0.636493,0.679383,8,80


## Linear Regression setup

Columns to encode = `[model_key, occupation_category, attended_university, gender]` and all except `occupation` can be *one-hot encoded*.
And the outcome variable would be a binary male/female (`gender`) variable. I will drop `response_number` and `profile_id` for the regression model.

Also need to group the entries by `profile_id` and average the entropy metrics (since the rest are already computed over groups).

In [18]:
# Encode gender to 0/1 for regression
df['gender_code'] = df['gender'].map({'female': 0, 'male': 1})

In [19]:
# Regression analysis for each metric using the requested formula
import statsmodels.formula.api as smf

reg_df = df.copy()

# Indicator columns
reg_df["is_female"] = (reg_df["gender"] == "female").astype(int)
reg_df["is_sft"] = (reg_df["model_key"] == "sft").astype(int)
reg_df["is_dpo"] = (reg_df["model_key"] == "dpo").astype(int)
reg_df["is_rlvr"] = (reg_df["model_key"] == "rlvr").astype(int)

if "temperature" not in reg_df.columns:
    reg_df["temperature"] = 0.0
    print("temperature column not found; set to 0.0 for all rows")

metric_cols = ["self_bleu_mean", "semantic_div_mean", "avg_mean_entropy_mean"]
metric_cols = [m for m in metric_cols if m in reg_df.columns]
if not metric_cols:
    raise ValueError("No metric columns found for regression.")

results = {}
for metric in metric_cols:
    formula = f"""
        {metric} ~ is_female + is_sft + is_dpo + is_rlvr
                    + is_female:is_sft + is_female:is_dpo + is_female:is_rlvr
                    + temperature
    """
    fit = smf.ols(formula, data=reg_df).fit()
    results[metric] = fit
    print(f"\n=== {metric} ===")
    print(fit.summary())

temperature column not found; set to 0.0 for all rows

=== self_bleu_mean ===
                            OLS Regression Results                            
Dep. Variable:         self_bleu_mean   R-squared:                       0.814
Model:                            OLS   Adj. R-squared:                  0.801
Method:                 Least Squares   F-statistic:                     60.08
Date:                Fri, 22 May 2026   Prob (F-statistic):           2.63e-32
Time:                        16:00:17   Log-Likelihood:                 252.18
No. Observations:                 104   AIC:                            -488.4
Df Residuals:                      96   BIC:                            -467.2
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------

c:\Users\manth\GitHub\occupational_bias_llms\env\.pixi\envs\default\Lib\site-packages\statsmodels\regression\linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])
c:\Users\manth\GitHub\occupational_bias_llms\env\.pixi\envs\default\Lib\site-packages\statsmodels\regression\linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])
c:\Users\manth\GitHub\occupational_bias_llms\env\.pixi\envs\default\Lib\site-packages\statsmodels\regression\linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])


In [13]:
# Per-occupation regression (SciPy least squares)
from scipy import linalg
import numpy as np

feature_cols = [
    c
    for c in ["self_bleu_mean", "semantic_div_mean", "avg_mean_entropy_mean"]
    if c in df.columns
 ]
if not feature_cols:
    raise ValueError("No metric columns found for regression.")

def fit_by_occupation(data):
    rows = []
    for occ, group in data.groupby("occupation", dropna=False):
        if group["gender_code"].nunique() < 2:
            continue
        X = group[feature_cols].copy()
        if "model_key" in group.columns:
            model_dummies = pd.get_dummies(group["model_key"], prefix="model", drop_first=True)
            X = pd.concat([X, model_dummies], axis=1)
        X = X.apply(pd.to_numeric, errors="coerce")
        y = pd.to_numeric(group["gender_code"], errors="coerce")
        valid = ~(X.isna().any(axis=1) | y.isna())
        X = X.loc[valid]
        y = y.loc[valid]
        if len(y) < len(X.columns) + 1:
            continue
        X = pd.concat([pd.Series(1.0, index=X.index, name="const"), X], axis=1)
        X_values = X.to_numpy(dtype=float)
        y_values = y.to_numpy(dtype=float)
        finite_mask = np.isfinite(X_values).all(axis=1) & np.isfinite(y_values)
        X_values = X_values[finite_mask]
        y_values = y_values[finite_mask]
        if len(y_values) == 0:
            continue
        coef, residuals, rank, s = linalg.lstsq(X_values, y_values)
        y_hat = X_values @ coef
        ss_res = ((y_values - y_hat) ** 2).sum()
        ss_tot = ((y_values - y_values.mean()) ** 2).sum()
        r2 = 1 - ss_res / ss_tot if ss_tot else float("nan")
        row = {"occupation": occ, "n": len(y_values), "r2": r2}
        for term, val in zip(X.columns, coef):
            row[f"coef_{term}"] = val
        rows.append(row)
    return pd.DataFrame(rows).sort_values("r2", ascending=False)

occ_results = fit_by_occupation(df)
occ_results.head(10)

,occupation,n,r2,coef_const,coef_self_bleu_mean,coef_semantic_div_mean,coef_model_dpo,coef_model_rlvr,coef_model_sft
1,author,8,0.983988,-25.137543,48.681773,23.112959,4.971479,5.280431,2.862640
6,nurse,8,0.978786,7.665820,-21.656550,3.667744,-2.095515,-2.003501,-2.029487
5,mechanic,8,0.943059,-34.935622,71.401600,22.145341,7.691158,6.365272,5.024938
8,professor,8,0.878416,22.821359,-58.108136,-0.802499,-7.372633,-6.868026,-5.292057
2,carpenter,8,0.866499,-8.564927,29.981784,-7.312901,3.499751,2.836522,1.766984
12,teacher,8,0.795392,-2.984252,-1.095819,10.662907,-0.269880,-0.446681,-0.947995
10,scientist,8,0.680027,23.966412,-53.757978,-7.568512,-6.573177,-5.990717,-4.788722
11,secretary,8,0.666802,12.764084,-30.057887,-1.592384,-3.269823,-2.518769,-1.511618
4,engineer,8,0.580488,-13.883318,30.253676,7.852549,3.176263,2.099849,2.212887
9,programmer,8,0.460959,-17.232756,34.803638,13.616230,2.420238,2.214833,1.620470


In [14]:
# Standardized coefficients per model_key (SciPy least squares)
from scipy import linalg
import numpy as np

feature_cols = [
    c
    for c in ["self_bleu_mean", "semantic_div_mean", "avg_mean_entropy_mean"]
    if c in df.columns
 ]
if not feature_cols:
    raise ValueError("No metric columns found for regression.")

def zscore_columns(frame, cols):
    stats = frame[cols].agg(["mean", "std"])
    out = frame.copy()
    for col in cols:
        std = stats.loc["std", col]
        if pd.isna(std) or std == 0:
            out[col] = np.nan
        else:
            out[col] = (out[col] - stats.loc["mean", col]) / std
    return out

def fit_by_model_key(data):
    rows = []
    for model_key, group in data.groupby("model_key", dropna=False):
        if group["gender_code"].nunique() < 2:
            continue
        group = group.dropna(subset=["gender_code"])
        group = zscore_columns(group, feature_cols)
        X = group[feature_cols].copy()
        X = X.apply(pd.to_numeric, errors="coerce")
        y = pd.to_numeric(group["gender_code"], errors="coerce")
        valid = ~(X.isna().any(axis=1) | y.isna())
        X = X.loc[valid]
        y = y.loc[valid]
        if len(y) < len(X.columns) + 1:
            continue
        X = pd.concat([pd.Series(1.0, index=X.index, name="const"), X], axis=1)
        X_values = X.to_numpy(dtype=float)
        y_values = y.to_numpy(dtype=float)
        finite_mask = np.isfinite(X_values).all(axis=1) & np.isfinite(y_values)
        X_values = X_values[finite_mask]
        y_values = y_values[finite_mask]
        if len(y_values) == 0:
            continue
        coef, residuals, rank, s = linalg.lstsq(X_values, y_values)
        y_hat = X_values @ coef
        ss_res = ((y_values - y_hat) ** 2).sum()
        ss_tot = ((y_values - y_values.mean()) ** 2).sum()
        r2 = 1 - ss_res / ss_tot if ss_tot else float("nan")
        for term, val in zip(X.columns, coef):
            rows.append({"model_key": model_key, "term": term, "coef": val, "n": len(y_values), "r2": r2})
    return pd.DataFrame(rows)

model_coef_df = fit_by_model_key(df)
model_coef_df.sort_values(["model_key", "term"])

,model_key,term,coef,n,r2
0,base,const,0.500000,26,0.116201
1,base,self_bleu_mean,0.036461,26,0.116201
2,base,semantic_div_mean,-0.156465,26,0.116201
3,dpo,const,0.500000,26,0.405913
4,dpo,self_bleu_mean,0.157272,26,0.405913
5,dpo,semantic_div_mean,0.408761,26,0.405913
6,rlvr,const,0.500000,26,0.342878
7,rlvr,self_bleu_mean,-0.002875,26,0.342878
8,rlvr,semantic_div_mean,0.297098,26,0.342878
9,sft,const,0.500000,26,0.268771


In [15]:
# Per-occupation gender differences in metrics, split by model_key
metric_cols = feature_cols
gender_filtered = df[df["gender"].isin(["male", "female"])].copy()

gender_means = (
    gender_filtered.groupby(["model_key", "occupation", "gender"], dropna=False)[metric_cols]
    .mean()
    .reset_index()
)

pivot = gender_means.pivot_table(
    index=["model_key", "occupation"],
    columns="gender",
    values=metric_cols,
    aggfunc="mean",
)

delta_cols = {}
for col in metric_cols:
    male_key = (col, "male")
    female_key = (col, "female")
    if male_key in pivot.columns and female_key in pivot.columns:
        delta_cols[f"delta_{col}_male_minus_female"] = pivot[male_key] - pivot[female_key]

delta_df = pd.DataFrame(delta_cols).reset_index()

counts = (
    gender_filtered.groupby(["model_key", "occupation", "gender"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .rename(columns={"male": "n_male", "female": "n_female"})
    .reset_index()
)

occ_gender_deltas = delta_df.merge(counts, on=["model_key", "occupation"], how="left")
occ_gender_deltas.head(10)

,model_key,occupation,delta_self_bleu_mean_male_minus_female,delta_semantic_div_mean_male_minus_female,n_female,n_male
0,base,artist,0.002590,-0.052972,1,1
1,base,author,0.026068,-0.011381,1,1
2,base,carpenter,0.031658,-0.068615,1,1
3,base,doctor,0.007791,-0.040397,1,1
4,base,engineer,0.010968,-0.057701,1,1
5,base,mechanic,0.026186,-0.052945,1,1
6,base,nurse,-0.035404,0.044151,1,1
7,base,politician,0.007058,-0.027856,1,1
8,base,professor,-0.007857,0.021511,1,1
9,base,programmer,0.040096,-0.089769,1,1
